# 4 · Look inside each gesture

**What this notebook does:** Many gestures have two bursts of speed: the hand moves *out* quickly, slows down or
briefly stops at the most meaningful moment, then moves *on* or *back* quickly. For each gesture found in
Notebook 3a, this notebook

* finds the **two biggest speed peaks**,
* marks the stretch **between** them (column `peaks`),
* marks the **slowest moment** between them (column `min`). This is a good candidate for the gesture's
  **apex**: the moment the gesture "hits" its target,
* marks every moment where the hand **changes direction**, left/right or up/down (columns ending in `_cross`).

<img src="../docs/inter-peak-interval.png" alt="The two peaks of a gesture, the stretch between them (grey) and the slowest moment (red)" width="420">

| You need | You get |
|---|---|
| `Events/Test_Video_events_found.csv` (from Notebook 3a) | `Gestures/Test_Video_gestures_found.csv`: the data from 3a plus the columns `peaks`, `min` and the direction-change columns (1 = yes, 0 = no). |

## Step 1 · Load the tools

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import find_peaks, peak_prominences

## Step 2 · Settings

In [ ]:
VIDEO_NAME = "Test_Video"

# Whose speed to use (the same body point as in Notebook 3a)
KEYPOINT = "right_wrist"

# Which gesture to show up close at the end of the notebook
GESTURE_TO_PLOT = 13

## Step 3 · Load the gestures from Notebook 3a

In [ ]:
data = pd.read_csv(f"../Events/{VIDEO_NAME}_events_found.csv")
speed_column = f"{KEYPOINT}_speed"
print(f"{data['gesture_id'].nunique()} gestures loaded")

## Step 4 · Find the two biggest peaks and the slowest moment between them

Inside each gesture we find all speed peaks and measure how much each one stands out from the dips around it
(its **prominence**: a small wiggle on the side of a big hill isn't very prominent). We keep the two most
prominent peaks, mark everything between them, and mark the slowest moment in between.

Gestures with only one peak get no marks.

In [ ]:
data["peaks"] = 0  # 1 = between the two biggest peaks of a gesture
data["min"] = 0    # 1 = the slowest moment between those peaks

for gesture_id, gesture in data.groupby("gesture_id"):
    speed = gesture[speed_column].to_numpy()
    peaks, _ = find_peaks(speed)
    if len(peaks) < 2:
        continue  # only one burst of speed: nothing to mark

    prominence = peak_prominences(speed, peaks)[0]
    # the two most prominent peaks, in time order, and the slowest moment between them
    first, last = sorted(peaks[np.argsort(prominence)[-2:]])
    slowest = first + np.argmin(speed[first:last + 1])

    rows = gesture.index  # where this gesture's rows are in the full table
    data.loc[rows[first:last + 1], "peaks"] = 1
    data.loc[rows[slowest], "min"] = 1

print(f"Marked a slowest moment in {data['min'].sum()} of {data['gesture_id'].nunique()} gestures")

## Step 5 · Mark changes of direction

The velocity columns from Notebook 2 (`_delta_x`, `_delta_y`) are positive when the hand moves one way and
negative when it moves the other way. Where the sign flips, the hand turned around.

In [ ]:
for axis in ["x", "y"]:
    velocity = f"{KEYPOINT}_delta_{axis}"
    data[velocity + "_cross"] = 0
    for gesture_id, gesture in data.groupby("gesture_id"):
        direction = np.sign(gesture[velocity].to_numpy())   # +1 or -1
        turns = np.where(np.diff(direction) != 0)[0]          # where the sign changes
        data.loc[gesture.index[turns], velocity + "_cross"] = 1

data[data["gesture_id"].notna()].head()

## Step 6 · Save

In [ ]:
Path("../Gestures").mkdir(exist_ok=True)
output_path = f"../Gestures/{VIDEO_NAME}_gestures_found.csv"
data.to_csv(output_path, index=False)
print("Saved:", output_path)

## Step 7 · Look at the whole video

Grey dots mark the stretch between the two biggest peaks of each gesture, red dots the slowest moment in between.
Use the slider to move through the video.

In [ ]:
between = data[data["peaks"] == 1]
slowest = data[data["min"] == 1]

fig = go.Figure()
fig.add_scatter(x=data["time_ms"], y=data[speed_column], name="speed", line_color="black")
fig.add_scatter(x=between["time_ms"], y=between[speed_column], name="between the two peaks",
                mode="markers", marker=dict(color="grey", size=5))
fig.add_scatter(x=slowest["time_ms"], y=slowest[speed_column], name="slowest moment",
                mode="markers", marker=dict(color="red", size=10))
fig.update_layout(title="The two biggest peaks and the slowest moment of each gesture",
                  xaxis_title="Time (ms)", yaxis_title="Speed (1 = fastest)",
                  xaxis_rangeslider_visible=True)
fig.show()

## Step 8 · Look at one gesture up close

Change `GESTURE_TO_PLOT` in the Settings (or right here) and run this cell again to look at another gesture.

In [ ]:
gesture = data[data["gesture_id"] == GESTURE_TO_PLOT]
if gesture.empty:
    last_gesture = int(data["gesture_id"].max())
    print(f"There is no gesture {GESTURE_TO_PLOT}. Choose a number from 1 to {last_gesture}.")
else:
    between = gesture[gesture["peaks"] == 1]
    slowest = gesture[gesture["min"] == 1]

    fig = go.Figure()
    fig.add_scatter(x=gesture["time_ms"], y=gesture[speed_column], name="speed",
                    line=dict(color="black", width=3))
    fig.add_scatter(x=between["time_ms"], y=between[speed_column], name="between the two peaks",
                    mode="markers", marker=dict(color="grey", size=9))
    fig.add_scatter(x=slowest["time_ms"], y=slowest[speed_column], name="slowest moment",
                    mode="markers", marker=dict(color="red", size=14))
    fig.update_layout(title=f"Gesture {GESTURE_TO_PLOT}", xaxis_title="Time (ms)",
                      yaxis_title=f"{KEYPOINT} speed", width=700, height=500)
    fig.show()

## Done!

Continue with **Notebook 5** to open these results in ELAN.